# 🎯 Nexus ALPR - Plate Detection Model Pipeline
**Enterprise Edition - Computer Vision Engineering Workflow**

This notebook handles the end-to-end training pipeline for the primary Object Detection model.
Its core purpose is to accurately localize and draw bounding boxes around Egyptian license plates in various environmental conditions (day/night, different angles, and occlusions).

### 📋 Pipeline Architecture:
1. **Environment Initialization:** Dependency resolution (including `sympy` versioning for stable ONNX exports).
2. **Data Ingestion:** Secure API pull from Roboflow for the annotated vehicle/plate dataset.
3. **Heavy Augmentation Training:** YOLOv8s fine-tuning utilizing advanced spatial and color augmentations (`mosaic`, `mixup`, `perspective`) for robust real-world detection.
4. **Production Export:** Freezing the best weights into ONNX format for deployment on the Nexus ALPR FastAPI backend.

In [ ]:
# ==========================================
# 📦 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==========================================
print("Installing core dependencies...")
!pip install -q ultralytics roboflow

# Pinning sympy to 1.12 to ensure stable ONNX graph exports in Colab environments
!pip install -q sympy==1.12
print("Dependencies installed successfully! 🚀")

## 📥 2. Dataset Acquisition
Fetching the primary vehicle and plate detection dataset from Roboflow.

In [ ]:
# ==========================================
# 📥 2. DATASET INGESTION (ROBOFLOW)
# ==========================================
from roboflow import Roboflow

# Initialize Roboflow via API Key
rf = Roboflow(api_key="**************")

# Target the detection workspace and project
project = rf.workspace("smartparkingsystem-jwynr").project("egyptian-vehaug-gwtkl")
dataset = project.version(1).download("yolov8")

print(f"Detection dataset successfully loaded at: {dataset.location}")

## 🧠 3. Model Training (YOLOv8s)
Training the detection model.
> **⚠️ ML Engineering Note:** We employ heavy augmentations here (`mosaic=1.0`, `mixup=0.2`) to force the model to learn plate localization in highly cluttered environments. `fliplr` and `flipud` are kept at `0.0` to maintain realistic physical orientations of vehicles and plates.

In [ ]:
# ==========================================
# 🧠 3. CORE DETECTION TRAINING PIPELINE
# ==========================================
from ultralytics import YOLO

# Initialize the YOLOv8 small architecture
model = YOLO('yolov8s.pt')

# Execute the training loop with complex augmentations
results = model.train(
    data='/content/egyptian-Vehaug-1/data.yaml',
    epochs=100,
    imgsz=640,               # Standard high-res input for bounding box detection
    batch=16,

    # 🚫 Disabled Flips (Preserves physical reality of vehicles/text)
    fliplr=0.0,
    flipud=0.0,

    # 🧩 Heavy Augmentations (For robustness against occlusion and tight parking spaces)
    mosaic=1.0,              # 100% probability of combining 4 images into 1
    mixup=0.2,               # 20% probability of image blending

    # 📐 Spatial & Lighting Adjustments
    degrees=10.0,            # Slight rotations to simulate angled cameras
    perspective=0.001,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # ⚙️ Training Callbacks
    patience=20,             # Early stopping if no improvement
    save_period=10           # Save checkpoints every 10 epochs
)

## 📦 4. Production Export (ONNX)
Packaging the best-performing weights into the ONNX format for seamless integration with the deployment environment.

In [ ]:
# ==========================================
# 📦 4. PRODUCTION EXPORT (ONNX)
# ==========================================

# 💡 Best Practice: Automatically load the best weights generated from this session
best_weights_path = 'runs/detect/train/weights/best.pt'
plate_model = YOLO(best_weights_path)

print("Exporting Plate Detection Model for Production...")

# Exporting to ONNX format.
# Dynamic=False enforces strict 640x640 tensor inputs for maximum inference speed on backend.
plate_model.export(format='onnx', imgsz=640, dynamic=False)

print("Export Complete! 🟢 The ONNX model is ready for the Nexus ALPR Backend.")